# Unlearning vs. Membership Inference: all experiments

Runs every target model of Table 1 (1a ... 3d) with every unlearning method (NegGrad, SCRUB, SFTC) with the unlearning learning rates selected by the learning-rate grid (`LEARNING_RATES`), and fills the paper table `tab:utility_mia_all`. All MIAs are evaluated after every unlearning epoch on the whole evaluation data.

**What happens for each target model**
1. The data splits used to train the saved models are rebuilt (seed 42, same split functions as `train_target_and_perform_MIA_attack.py`).
2. The saved models are loaded from `models/`: target, retrained-from-scratch baseline, and the shadow-model attack (MIA-1). Nothing is retrained.
3. Each method unlearns the target for `EPOCHS` epochs. After **every** epoch it measures test accuracy and the forget/retain AUC of
   - **MIA-1**: shadow-model + RandomForest attack (saved `.jolib` model)
   - **MIA-2**: loss-threshold attack
   - **MIA-3**: offline **LiRA** (Carlini et al., 2022), with 16 reference models per target model (Step 1)
4. **Best** = epoch minimising `|MIA-1 forget AUC − Retr.| + |Test Acc − Retr. Test Acc|`; **Final** = last epoch. AUC is reported as max(AUC, 1−AUC).

**Reproducibility**
- Fixed seeds, deterministic cuDNN/cuBLAS, every run reseeded at its start, so results don't depend on run order.
- Each target model runs in its own subprocess (clean state, and memory is freed between experiments).
- Results are cached in `results_lira/<exp>/<method>.json` together with their config and software versions. Re-running the notebook skips finished runs, so it **resumes after an interruption**. Changing a learning rate (or epochs/seed) re-runs only that run. To force a re-run, delete its JSON file.

**Run it without keeping the notebook open** (from the project folder):
```
jupyter nbconvert --to notebook --execute --inplace --ExecutePreprocessor.timeout=-1 run_all_experiments.ipynb
```
Outputs: `results/summary.csv`, `results/table_utility_mia_all.tex`, and per-epoch curves in `results/<exp>/<method>_epochs.csv`.

In [ ]:
import os
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")  # deterministic cuBLAS; set before CUDA starts

import json
import pandas as pd
import experiment_runner as er

env = er.environment_info()
print(json.dumps(env, indent=2))
if env["uncommitted_python_changes"]:
    print("Note: there are uncommitted .py changes; commit them so results can be tied to a git commit.")

## Configuration

`LEARNING_RATES`: one unlearning learning rate per target model and method, selected by the learning-rate grid (`results_lrgrid/table_lr_grid.tex`). Set a value to `None` to skip that run (it shows as `--` in the table). Changing a value re-runs only that run.

In [ ]:
SEED = 42
EPOCHS = 30          # unlearning epochs
BATCH_SIZE = 32
MOMENTUM = 0.9       # SGD momentum, as in main.py
RESULTS_DIR = "results_lira"   # results/ holds the previous runs (MIA-3 = max confidence)
MIA3 = "lira"                  # MIA-3 = offline LiRA

EXPERIMENTS_TO_RUN = er.ALL_EXPERIMENTS   # e.g. ["1d", "2d", "3d"] to run a subset
METHODS = er.METHODS                       # ["neg_grad", "scrub", "sftc"]

LEARNING_RATES = {   # selected by the learning-rate grid (results_lrgrid/table_lr_grid.tex)
    "1a": {"neg_grad": 1e-3, "scrub": 1e-5, "sftc": 1e-3},
    "1b": {"neg_grad": 1e-2, "scrub": 1e-4, "sftc": 1e-2},   # NegGrad: best at epoch 1, diverges after
    "1c": {"neg_grad": 1e-3, "scrub": 1e-4, "sftc": 1e-2},
    "1d": {"neg_grad": 1e-4, "scrub": 1e-3, "sftc": 1e-2},
    "2a": {"neg_grad": 1e-4, "scrub": 1e-5, "sftc": 1e-3},
    "2b": {"neg_grad": 1e-3, "scrub": 1e-4, "sftc": 1e-3},
    "2c": {"neg_grad": 1e-3, "scrub": 1e-4, "sftc": 1e-3},
    "2d": {"neg_grad": 1e-3, "scrub": 1e-4, "sftc": 1e-2},
    "3a": {"neg_grad": 1e-4, "scrub": 1e-4, "sftc": 1e-2},   # NegGrad: selected model collapsed (test acc 0.15)
    "3b": {"neg_grad": 1e-4, "scrub": 1e-3, "sftc": 1e-2},
    "3c": {"neg_grad": 1e-4, "scrub": 1e-7, "sftc": 1e-2},   # SCRUB: flat region, no LR changes the model
    "3d": {"neg_grad": 1e-4, "scrub": 1e-6, "sftc": 1e-2},   # SCRUB: flat region, no LR changes the model
}

missing = [(e, er.METHOD_NAMES[m]) for e in EXPERIMENTS_TO_RUN for m in METHODS if LEARNING_RATES[e][m] is None]
print(f"{len(missing)} runs have no learning rate yet and will be skipped:")
print(", ".join(f"{e} {m}" for e, m in missing) or "none")

## Step 1: LiRA reference models (for MIA-3)
For every target model, 16 reference models of the same architecture are trained on random subsets of the shadow data (disjoint from the target data), with the target's training settings (`lira.TRAIN_CONFIG`). Only missing models are trained; each one is saved as soon as it is trained (`models/lira/<exp>/`). The tabular models take minutes, the ResNets (3a, 3b) hours.

In [ ]:
import lira
import train_lira_references as tlr

failed_lira = tlr.run(EXPERIMENTS_TO_RUN, lira.N_REFERENCE)

## Step 2: run all experiments
One line per epoch is printed. Finished runs are loaded from `results_lira/` instead of re-run.

In [ ]:
failed = er.run_all(EXPERIMENTS_TO_RUN, METHODS, LEARNING_RATES, epochs=EPOCHS, batch_size=BATCH_SIZE,
                    momentum=MOMENTUM, seed=SEED, results_dir=RESULTS_DIR, mia3=MIA3)

## Results

In [ ]:
results = er.load_results(RESULTS_DIR)
summary = er.summary_dataframe(results)
summary.to_csv(os.path.join(RESULTS_DIR, "summary.csv"), index=False)
summary.round(3)

## LaTeX table
Written to `results_lira/table_utility_mia_all.tex`; include it with `\input{...}`. Final/Best test accuracies below 25% of the retrained model's are bolded (model collapse).

In [ ]:
tex = er.make_latex_table(results)
with open(os.path.join(RESULTS_DIR, "table_utility_mia_all.tex"), "w") as f:
    f.write(tex)
print(tex)

## Optional: per-epoch curves for one run

In [ ]:
import matplotlib.pyplot as plt

EXP, METHOD = "3c", "sftc"   # pick any finished run
path = os.path.join(RESULTS_DIR, EXP, f"{METHOD}_epochs.csv")
if os.path.exists(path):
    h, r = pd.read_csv(path), results[EXP][METHOD]
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(h["epoch"], h["test_acc"], label="Test acc")
    axes[0].axhline(r["retrain"]["test_acc"], ls="--", c="gray", label="Retrained")
    for a, c in zip(er.ATTACKS, ["C0", "C1", "C2"]):
        axes[1].plot(h["epoch"], h[f"forget_{a}_auc"], c=c, label={"mia1": "MIA-1", "mia2": "MIA-2", "mia3": "MIA-3 (LiRA)"}[a])
        axes[1].axhline(r["retrain"][f"forget_{a}_auc"], ls="--", c=c)
    for ax, t in zip(axes, ["Test accuracy", "Forget-set MIA AUC (dashed: retrained)"]):
        ax.axvline(r["best_epoch"], c="k", ls=":", label="Best epoch")
        ax.set_title(t); ax.set_xlabel("Unlearning epoch"); ax.legend(); ax.grid(alpha=.3)
    fig.suptitle(f"{EXP} {er.METHOD_NAMES[METHOD]} (lr={r['config']['lr']:g})")
    plt.show()
else:
    print(f"No results yet for {EXP} {METHOD}")